In [21]:
file_path = "../data/attention is all you need.pdf"

import pymupdf4llm
import pathlib
from pprint import pprint

md_text = pymupdf4llm.to_markdown(file_path)

pathlib.Path("attention_is_all_you_need.md").write_bytes(md_text.encode())

42413

In [22]:
chunks = []
for i in range(0, len(md_text), 500):
    chunks.append(md_text[i : i + 500])

len(chunks)

85

In [23]:
chunks[0]

'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. \n\n# **Attention Is All You Need** \n\n**Ashish Vaswani** _[∗]_ **Noam Shazeer** _[∗]_ **Niki Parmar** _[∗]_ **Jakob Uszkoreit** _[∗]_ Google Brain Google Brain Google Research Google Research `avaswani@google.com noam@google.com nikip@google.com usz@google.com` \n\n**Llion Jones** _[∗]_ **Aidan N. Gomez** _[∗†]_ **Łukasz Kaiser*'

In [24]:
from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import PointStruct

client = QdrantClient(url="http://localhost:6333")

collection_name = "one_paper"
embedding_model_dimensions = 384

dense_model = "BAAI/bge-small-en"
sparse_model = "qdrant/bm25"

In [25]:
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "dense_vector": models.VectorParams(
                size=embedding_model_dimensions, distance=models.Distance.COSINE
            )
        },
        sparse_vectors_config={
            "bm25_sparse_vector": models.SparseVectorParams(
                modifier=models.Modifier.IDF
            )
        },
    )

In [26]:
# client.delete_collection(collection_name=collection_name)

In [27]:
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding

dense_encoder = SentenceTransformer(dense_model)
dense_embeddings = dense_encoder.encode(chunks, show_progress_bar=True)

bm25_encoder = SparseTextEmbedding(model_name=sparse_model)
sparse_embeddings = list(bm25_encoder.embed(chunks))

print(f"Dense shape: {dense_embeddings.shape}")
print(f"Sparse vectors: {len(sparse_embeddings)}")

Batches: 100%|██████████| 3/3 [00:02<00:00,  1.16it/s]

Dense shape: (85, 384)
Sparse vectors: 85


In [28]:
points = []
for idx, (chunk, dense_vec, sparse_vec) in enumerate(
    zip(chunks, dense_embeddings, sparse_embeddings)
):
    point = PointStruct(
        id=idx + 1,
        payload={
            "text": chunk,
            "arxiv_id": "1706.03762",
            "chunk_idx": idx,
        },
        vector={
            "dense_vector": dense_vec.tolist(),
            "bm25_sparse_vector": models.SparseVector(
                indices=sparse_vec.indices.tolist(), values=sparse_vec.values.tolist()
            ),
        },
    )
    points.append(point)

client.upload_points(collection_name=collection_name, points=points, batch_size=8)
print(f"Uploaded {len(points)} chunks")

Uploaded 85 chunks


In [29]:
query = "How does multi-head attention work?"

dense_query_vec = dense_encoder.encode(query)
sparse_query_vec = next(bm25_encoder.embed([query]))

In [30]:
results = client.query_points(
    collection_name=collection_name,
    prefetch=[
        models.Prefetch(query=dense_query_vec.tolist(), using="dense_vector", limit=5),
        models.Prefetch(
            query=models.SparseVector(
                indices=sparse_query_vec.indices.tolist(),
                values=sparse_query_vec.values.tolist(),
            ),
            using="bm25_sparse_vector",
            limit=5,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=5,
)


pprint(results.points)

[ScoredPoint(id=27, version=4, score=0.6666667, payload={'text': 'head attention with full dimensionality. \n\n## **3.2.3 Applications of Attention in our Model** \n\nThe Transformer uses multi-head attention in three different ways: \n\n- In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-seque', 'arxiv_id': '1706.03762', 'chunk_idx': 26}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=5, version=1, score=0.5, payload={'text': 'ish, with Illia, designed and implemented the first Transformer models and has been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head attention and the parameter-free position representation and became the other person i